## AE33 TCP/IP Communication Guide

This notebook demonstrates how to communicate with the **AE33 Aethalometer** using its TCP/IP interface for real-time data acquisition and control.

**License:** Aerosol Magee Scientific Software License. See LICENSE file for full terms.

In [17]:
from io import StringIO
import re

import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp

In [18]:
# IP of the instrument - change it to the actual IP address of your AE33 instrument
instrument_ip = '10.10.10.224'

In [19]:
# default column names
COLUMNS_AE33_DATA = ['SerialNumber', 'ID', 'StartTime', 'EndTime', 'SetupID', 'SetupTimestamp',
                     'Ref1', 'Sens11', 'Sens12', 'Ref2', 'Sens21', 'Sens22',
                     'Ref3', 'Sens31', 'Sens32', 'Ref4', 'Sens41', 'Sens42',
                     'Ref5', 'Sens51', 'Sens52', 'Ref6', 'Sens61', 'Sens62',
                     'Ref7', 'Sens71', 'Sens72',
                     'BC11', 'BC12', 'BC1', 'BC21', 'BC22', 'BC2',
                     'BC31', 'BC32', 'BC3', 'BC41', 'BC42', 'BC4',
                     'BC51', 'BC52', 'BC5', 'BC61', 'BC62', 'BC6',
                     'BC71', 'BC72', 'BC7',
                     'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'BB',
                     'Pressure', 'Temp', 'Flow1', 'Flow2', 'FlowC',
                     'T_controller', 'T_supply', 'T_LED', 'ControllerStatus',
                     'LEDStatus', 'DetectorStatus', 'ValveStatus', 'Status',
                     'TapeAdvanceCount', 'TapeAdvanceLeft', 'CPU', 'DiskSpace', 'NumConnections']
COLUMNS_AE33_EXTERNAL_DEVICE_DATA = ['SerialNumber', 'ID', 'DataID', 'DeviceID', 'DeviceData']
COLUMNS_AE33_SETUP = ['serial', 'ID', 'SerialNumber', 'Timestamp', 'FirmwareVer', 'SoftwareVer',
                      'DataCenterIP', 'AutoConnect', 'InletFilter',
                      'Timebase', 'SG1', 'SG2', 'SG3', 'SG4', 'SG5', 'SG6', 'SG7',
                      'C', 'Area', 'Zeta', 'Aff', 'Abb', 'Pressure', 'Temp',
                      'ATNf1', 'ATNf2', 'Kmax', 'Kmin', 'Flow', 'FlowRepStd',
                      'PumpPresetValue', 'FlowFormulaA0', 'FlowFormulaA1',
                      'FlowFormulaB0', 'FlowFormulaB1', 'FlowFormulaC0',
                      'FlowFormulaC1', 'FlowFormulaD', 'FlowFormulaE',
                      'FlowFormulaF', 'TAtype', 'TAatnMax', 'TAinterval',
                      'TAtime', 'TapeRightFormulaK', 'TapeRightFormulaN',
                      'TapeLeftFormulaK', 'TapeLeftFormulaN', 'WarmUpInterval',
                      'AutoTestEnabled', 'AutoTestType', 'AutoTestDay', 'AutoTestTime',
                      'MeasureTimeStamp', 'HomeInfo', 'Display', 'About', 'DST', 'TimeZone',
                      'TapeAdvanceAdjust', 'ExternalID', 'BHparamID', 'TimeSync', 'DHCP',
                      'InstrumentIP', 'SubnetMask', 'Gateway', 'Baud', 'NTPserver']

In [20]:
### command HELLO
command_ae33 = 'HELLO\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)

AE33>Aethalometer information
Server type: AE33
Instrument serialnumber: AE33-S06-00565
Time: 11/Jun/2026 07:49:53
Database version: 1.7.3
Connected Clients: 3
ID: 1225572  IP: 10.10.10.253:49741  Time: 26 May 2026 15:47:07  Streaming: True
ID: 1342404  IP: 10.10.10.253:59184  Time: 10 Jun 2026 08:36:36  Streaming: True
ID: 1350124  IP: 192.168.45.37:54513  Time: 11 Jun 2026 07:49:53  Streaming: False

AE33>


In [21]:
### command MAXID Data
command_ae33 = 'MAXID Data\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)

# # parse data
max_data_id = int(re.search(r'AE33>(\d+)', received_text).group(1))
print()
print(max_data_id)


AE33>3159526
AE33>

3159526


In [22]:
### command FETCH Data - collect last 5 rows from table Data
command_ae33 = f'FETCH Data {max_data_id-5} {max_data_id}\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_data = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_DATA)
print()
print(df_ae33_data)

AE33>AE33>AE33-S06-00565|3159521|6/11/2026 7:43:00 AM|6/11/2026 7:44:00 AM|26|7/29/2025 7:24:13 AM|929761|611029|681436|902839|639792|701577|917594|663058|696286|928621|667585|698155|924317|776762|828257|796521|871543|930544|876621|888703|930863|42|28|44|39|12|40|13|-21|14|40|5|41|27|3|28|37|-21|38|39|-20|40|0.006439992|0.006718565|0.007553311|0.008686225|0.009296549|0.01159205|0.01154316|1.4|101325.0|21.1|3622|1247|4869|32.0|39.0|33.0|0|10|10|0|0|6066|284|6|5241|2
AE33>AE33-S06-00565|3159522|6/11/2026 7:44:00 AM|6/11/2026 7:45:00 AM|26|7/29/2025 7:24:13 AM|929627|610954|681370|902703|639696|701492|917452|662966|696209|928363|667410|697983|924125|776602|828104|796306|871318|930316|876412|888489|930662|-18|-117|-19|-1|-93|-1|-19|-154|-20|-21|-124|-22|-3|-102|-3|-24|-147|-25|5|-143|5|0.006439992|0.006718565|0.007553311|0.008686225|0.009296549|0.01159205|0.01154316|0|101325.0|21.1|3622|1247|4869|32.0|38.0|33.0|0|10|10|0|0|6066|284|6|5241|2
AE33>AE33-S06-00565|3159523|6/11/2026 7:45:00 AM|

In [23]:
### collect data from Setup table
command_ae33 = f'FETCH SETUP\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_setup = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                            sep='|',
                            names=COLUMNS_AE33_SETUP)
# drop duplicated serial column from setup data
df_ae33_setup = df_ae33_setup.drop(columns=['serial'], errors='ignore')
print()
print(df_ae33_setup)

AE33>AE33-S06-00565|1|AE33-S06-00565|10/30/2020 4:47:54 PM|533|1.4.9.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|120|12|3/13/2017 10:57:40 AM|1.04204201698303|1.88889074325562|1.12658226490021|-45.7848091125488|1|1|0|2|1/1/2014 12:00:00 AM|1|0|1|0|0|Coordinated Universal Time|10|1|1|1|1|192.168.0.2|255.255.255.0|192.168.0.1|115200|pool.ntp.org
AE33>AE33>AE33-S06-00565|13|AE33-S06-00565|9/14/2021 7:23:04 AM|540|1.5.0.1|10.10.10.253:8007|1|0|60|18.47|14.54|13.14|11.58|10.35|7.77|7.19|1.39|0.785|0.02|1|2|101325|25.00|10|30|0.015|-0.005|2000|1|585|-1952.80444335937|-2505.5458984375|11.6349792480469|12.9688386917114|0.000753700849600136|-0.000350014335708693|166.16487121582|0.0840134471654892|-4.06122467211389E-07|1|120|12|3/13/2017 10:57

In [24]:
### collect data from ExtDeviceData table - get MAXID and then fetch last 5 rows

command_ae33 = 'MAXID ExtDeviceData\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)
max_ext_device_data_id = int(re.search(r'AE33>(\d+)', received_text).group(1))

command_ae33 = f'FETCH ExtDeviceData {max_ext_device_data_id-5} {max_ext_device_data_id}\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command_ae33)
print(received_text)

# in the end, you can parse data, for example convert it into pandas dataframe
df_ae33_device_data = pd.read_csv(StringIO(received_text.replace('AE33>', '').strip()),
                           sep='|',
                           names=COLUMNS_AE33_EXTERNAL_DEVICE_DATA)
print()
print(df_ae33_device_data)

AE33>4442967
AE33>
AE33>AE33-S06-00565|4442962|3159522|15|100
AE33>AE33>AE33-S06-00565|4442963|3159523|15|100
AE33>AE33-S06-00565|4442964|3159524|15|100
AE33>AE33-S06-00565|4442965|3159525|15|100
AE33>AE33-S06-00565|4442966|3159526|15|100
AE33>AE33-S06-00565|4442967|3159527|15|100
AE33>

     SerialNumber       ID   DataID  DeviceID  DeviceData
0  AE33-S06-00565  4442962  3159522        15         100
1  AE33-S06-00565  4442963  3159523        15         100
2  AE33-S06-00565  4442964  3159524        15         100
3  AE33-S06-00565  4442965  3159525        15         100
4  AE33-S06-00565  4442966  3159526        15         100
5  AE33-S06-00565  4442967  3159527        15         100
